# WarBias unlearning setup

Install [requirements-unlearning.txt](../requirements-unlearning.txt) with the [setup guide](SETUP.md). Configure the checkpoint root, WarBias direction bundles and output root below.

In [1]:
import subprocess
import sys
from pathlib import Path

repo = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/unlearning/models.json").is_file()
)
sys.path.insert(0, str(repo / "src"))
from fairforget.unlearning.warbias_plan import selected_models  # noqa: E402

models = selected_models(repo)
for model in models:
    print(model["id"], model["repository"], model["revision"])
print("Models:", len(models))

gemma3_12b_it google/gemma-3-12b-it 96b6f1eccf38110c56df3a15bffe176da04bfd80
lapa12b_instruct lapa-llm/lapa-v0.1.2-instruct 2969fd997f07b33727c6f10795d78ef23cc5c037
mamay12b_v2 INSAIT-Institute/MamayLM-Gemma-3-12B-IT-v2.0 3cb02013b6b37cfe10f4b3da0fcddc1acef7f244
Models: 3


## Local paths

The setup prepares four separate WarBias bundles: IDP/Ukrainian, IDP/English, veteran/Ukrainian and veteran/English. Enable `DOWNLOAD_MODELS` to download the three selected model revisions. Hugging Face access uses the `HF_TOKEN` environment variable.

In [2]:
sys.path.insert(0, str(repo / "src"))
from fairforget.unlearning.warbias_data import PROTOCOL, default_root

MODELS_ROOT = repo / "models"
WARBIAS_DATA = default_root(repo)
OUTPUT_ROOT = repo / "outputs/unlearning" / PROTOCOL
DOWNLOAD_MODELS = False

In [3]:
command = [
    sys.executable,
    str(repo / "scripts/setup_unlearning.py"),
    "--models-root",
    str(MODELS_ROOT),
    "--warbias-data",
    str(WARBIAS_DATA),
    "--output-root",
    str(OUTPUT_ROOT),
]
if DOWNLOAD_MODELS:
    command.append("--download-models")
completed = subprocess.run(command, cwd=repo, check=True, capture_output=True, text=True)
print(completed.stdout.replace(str(repo), "<repo>"), end="")

gemma3_12b_it google/gemma-3-12b-it 96b6f1eccf38110c56df3a15bffe176da04bfd80
lapa12b_instruct lapa-llm/lapa-v0.1.2-instruct 2969fd997f07b33727c6f10795d78ef23cc5c037
mamay12b_v2 INSAIT-Institute/MamayLM-Gemma-3-12B-IT-v2.0 3cb02013b6b37cfe10f4b3da0fcddc1acef7f244
{
  "models_available": {
    "gemma3_12b_it": false,
    "lapa12b_instruct": false,
    "mamay12b_v2": false
  },
  "data_available": {
    "warbias_application": true
  },
  "all_inputs_available": false
}
Setup saved. Selected models: 3.
Use --download-models or point --models-root to the staged checkpoints.


## Training

Run [BiasUnlearn](biasunlearn.ipynb) and [WGA+KL](wga_kl.ipynb). Each method has 12 independent one-epoch runs: three models in four group/language directions, 24 runs in total. Every run starts from the pinned base checkpoint and trains on one group in one language. Each adapter is evaluated only on its training group and language. Cross-group and cross-language checks are deferred. See the [WarBias workflow](WARBIAS_UNLEARNING.md).
